# DotWall world model with the fine-tuned I-JEPA encoder

This notebook regenerates the episodes from the **existing** layout manifest, embeds every frame with the selected fine-tuned encoder, trains a new proprioceptive world model, and plans on the manifest's four unseen test layouts. It uses the same world-model and CEM settings as the earlier multi-layout run. It does not run other models.

Raw episodes, latent cache, audit, rollout reports, and planning results stay under `/content` in this Colab runtime. **No cache shards are written to Drive.** The world-model trainer writes `best.pt`, `last.pt`, and a small `test_metrics.pt` into a new Drive run folder. If the runtime resets, rerun generation and embedding before resuming the world model. Budget roughly 110 GiB of local disk for raw frames and embeddings together.

Run the cells in order on a GPU runtime. The first episode and first planning episode are functional smoke checks; their metrics do not characterize the full experiment.


## 1. Set up the pinned code and GPU

In [ ]:
from pathlib import Path
from google.colab import drive
import os
import subprocess
import sys

drive.mount("/content/drive")

REPO = Path("/content/i-jepa-wm")
DINO_ROOT = Path("/content/dino_wm")
BRANCH = "wall-layout-generalization"
DINO_COMMIT = "0a9492fa12044b852ae9e001cc74604b79c8bb0c"

if not REPO.exists():
    subprocess.run([
        "git", "clone", "--branch", BRANCH, "--single-branch",
        "https://github.com/kmylonas/i-jepa-wm.git", str(REPO),
    ], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO), "switch", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)

if not DINO_ROOT.exists():
    subprocess.run([
        "git", "clone", "https://github.com/gaoyuezhou/dino_wm.git",
        str(DINO_ROOT),
    ], check=True)
subprocess.run(["git", "-C", str(DINO_ROOT), "checkout", DINO_COMMIT], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "gym==0.26.2", "omegaconf", "scipy", "pyyaml", "imageio", "imageio-ffmpeg",
], check=True)

os.chdir(REPO)
sys.path.insert(0, str(REPO))

import torch
assert torch.cuda.is_available(), "Choose a Colab GPU runtime first"
assert (DINO_ROOT / "env/wall/envs/wall.py").is_file()
print("GPU:", torch.cuda.get_device_name(0))
print("Repository commit:", subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"], text=True,
).strip())

# All project commands run against this checkout.
def run(command):
    print("$", " ".join(map(str, command)), flush=True)
    subprocess.run([str(part) for part in command], cwd=REPO, check=True)


## 2. Select the existing manifest and exported encoder

The manifest and environment configuration are read from the earlier multi-layout dataset. The notebook copies the environment configuration locally so its DINO-WM path points to this runtime. The fine-tuned encoder is copied locally and checked against the manifest used for fine-tuning. Edit `ENCODER_DRIVE` only if you exported it elsewhere.


In [ ]:
import hashlib
import json
import shutil
from src.mylonas_ijepa_wm.wall_layout_manifest import load_manifest, manifest_digest

DATA_DRIVE = Path("/content/drive/MyDrive/datasets/wall_layout_generalization")
ENCODER_DRIVE = DATA_DRIVE / "ijepa_finetune_4epoch_checkpoints_v1/jepa-best.encoder-only.pth.tar"
MANIFEST_DRIVE = DATA_DRIVE / "episode_manifest.json"
ENV_DRIVE = DATA_DRIVE / "environment.json"

LOCAL_ROOT = Path("/content/wall_finetuned_encoder_wm")
RAW_DIR = LOCAL_ROOT / "raw"
CACHE_DIR = LOCAL_ROOT / "cache_dense"
RESULTS_DIR = LOCAL_ROOT / "results"
RUN_DIR = DATA_DRIVE / "world_model_finetuned_encoder_v1"
MANIFEST = LOCAL_ROOT / "episode_manifest.json"
ENV_CONFIG = LOCAL_ROOT / "environment.json"
ENCODER_LOCAL = LOCAL_ROOT / "fine_tuned_encoder.pth.tar"

for path in (MANIFEST_DRIVE, ENV_DRIVE, ENCODER_DRIVE):
    assert path.is_file(), f"Missing input: {path}"
for path in (LOCAL_ROOT, RAW_DIR, CACHE_DIR, RESULTS_DIR, RUN_DIR):
    path.mkdir(parents=True, exist_ok=True)

shutil.copy2(MANIFEST_DRIVE, MANIFEST)
settings = json.loads(ENV_DRIVE.read_text())
settings["dino_wm_root"] = str(DINO_ROOT)
ENV_CONFIG.write_text(json.dumps(settings, indent=2))
if not ENCODER_LOCAL.exists():
    shutil.copy2(ENCODER_DRIVE, ENCODER_LOCAL)

manifest = load_manifest(MANIFEST)
manifest_hash = manifest_digest(manifest)
exported = torch.load(ENCODER_LOCAL, map_location="cpu", weights_only=True, mmap=True)
assert exported["wall_run_identity"]["manifest_sha256"] == manifest_hash, (
    "Fine-tuned encoder and episode manifest differ"
)
assert exported["epoch"] == 1, "Expected the selected epoch-1 encoder"
assert "encoder" in exported and exported["encoder"]
del exported

# Guard local --resume against accidentally mixing encoders in one cache.
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

encoder_hash = sha256_file(ENCODER_LOCAL)
cache_identity = {"encoder_sha256": encoder_hash, "manifest_sha256": manifest_hash,
                  "embedding_frame_stride": 1}
cache_identity_path = CACHE_DIR / "embedding_identity.json"
if cache_identity_path.exists():
    assert json.loads(cache_identity_path.read_text()) == cache_identity, (
        "This local cache belongs to another encoder or manifest"
    )
else:
    assert not list(CACHE_DIR.glob("latents/episode_*.npy")), (
        "Existing latents have no encoder identity; choose a fresh CACHE_DIR"
    )
    cache_identity_path.write_text(json.dumps(cache_identity, indent=2))

episode_count = len(manifest["episodes"])
assert episode_count == 2600, f"Unexpected episode count: {episode_count}"
raw_episode_bytes = (manifest["steps_per_episode"] + 1) * 224 * 224 * 3
latent_episode_bytes = (manifest["steps_per_episode"] + 1) * 256 * 1280 * 2
missing_raw = sum(not (RAW_DIR / f"episode_{i:04d}.pt").is_file()
                  for i in range(episode_count))
missing_cache = sum(
    not (CACHE_DIR / "latents" / f"episode_{i:04d}.npy").is_file()
    or not (CACHE_DIR / "trajectories" / f"episode_{i:04d}.pt").is_file()
    for i in range(episode_count)
)
remaining_bytes = missing_raw * raw_episode_bytes + missing_cache * latent_episode_bytes
free_bytes = shutil.disk_usage("/content").free
print("Manifest:", manifest["episode_counts"], manifest_hash[:12])
print("Encoder SHA-256:", encoder_hash[:16])
print(f"Estimated additional raw + latent storage: {remaining_bytes / 2**30:.1f} GiB")
print(f"Free /content disk: {free_bytes / 2**30:.1f} GiB")
assert free_bytes > remaining_bytes + 10 * 2**30, (
    "Insufficient /content space for remaining frames, embeddings, and working files"
)


## 3. Smoke check: generate and embed one episode

This checks the renderer, exported encoder, embedding shape, and trajectory alignment before generating the full dataset.


In [ ]:
import numpy as np

run([
    sys.executable, "-u", "-m", "src.mylonas_ijepa_wm.generate_episodes",
    "--env", "wall", "--env-config", ENV_CONFIG,
    "--episode-manifest", MANIFEST, "--output-dir", RAW_DIR,
    "--steps-per-episode", "50", "--start-index", "0", "--end-index", "1",
    "--resume",
])
run([
    sys.executable, "-u", "-m", "src.mylonas_ijepa_wm.embed_frames",
    "--env", "wall", "--env-config", ENV_CONFIG,
    "--input-dir", RAW_DIR, "--output-dir", CACHE_DIR,
    "--checkpoint-path", ENCODER_LOCAL, "--embedding-frame-stride", "1",
    "--start-index", "0", "--end-index", "1", "--encoder-batch-size", "4",
    "--resume",
])
latent = np.load(CACHE_DIR / "latents/episode_0000.npy", mmap_mode="r")
trajectory = torch.load(
    CACHE_DIR / "trajectories/episode_0000.pt", map_location="cpu", weights_only=True,
)
assert latent.shape == (51, 256, 1280) and latent.dtype == np.float16
assert tuple(trajectory["frame_indices"].tolist()) == tuple(range(51))
assert trajectory["manifest_sha256"] == manifest_hash
print("Smoke passed: 51 aligned frames, latent shape", latent.shape)


## 4. Regenerate and audit the full episode collection

The same manifest gives the same train, validation, and held-out layout assignments as before. Raw episodes and the coverage report remain in `/content`.


In [ ]:
run([
    sys.executable, "-u", "-m", "src.mylonas_ijepa_wm.generate_episodes",
    "--env", "wall", "--env-config", ENV_CONFIG,
    "--episode-manifest", MANIFEST, "--output-dir", RAW_DIR,
    "--steps-per-episode", "50", "--resume",
])
assert len(list(RAW_DIR.glob("episode_*.pt"))) == episode_count
coverage_path = RESULTS_DIR / "coverage.json"
run([
    sys.executable, "-u", "-m", "src.mylonas_ijepa_wm.audit_wall_dataset",
    "--raw-dir", RAW_DIR, "--episode-manifest", MANIFEST,
    "--output", coverage_path, "--require-coverage",
])
print("Coverage passes:", json.loads(coverage_path.read_text())["coverage_pass"])


## 5. Embed all episodes locally

This loop writes only local `latents/` and `trajectories/` files. It does not create tar archives or copy embeddings to Drive. `--resume` is safe within this cache because section 2 checks the encoder and manifest hashes.


In [ ]:
import time

for start in range(0, episode_count, 100):
    end = min(start + 100, episode_count)
    t0 = time.monotonic()
    run([
        sys.executable, "-u", "-m", "src.mylonas_ijepa_wm.embed_frames",
        "--env", "wall", "--env-config", ENV_CONFIG,
        "--input-dir", RAW_DIR, "--output-dir", CACHE_DIR,
        "--checkpoint-path", ENCODER_LOCAL,
        "--embedding-frame-stride", "1",
        "--start-index", str(start), "--end-index", str(end),
        "--encoder-batch-size", "4", "--resume",
    ])
    print(f"Embedded episodes {start}–{end - 1} in "
          f"{(time.monotonic() - t0) / 60:.1f} min", flush=True)

missing = [i for i in range(episode_count)
           if not (CACHE_DIR / "latents" / f"episode_{i:04d}.npy").is_file()
           or not (CACHE_DIR / "trajectories" / f"episode_{i:04d}.pt").is_file()]
assert not missing, f"Missing local cache pairs: {missing[:10]}"
print(f"Local latent/trajectory pairs: {episode_count}/{episode_count}")


## 6. Train a new world model

The encoder stays frozen during world-model training. The new Drive folder keeps `best.pt` and `last.pt` for resume; the trainer also writes a small `test_metrics.pt` there after completion. Raw episodes and embeddings stay local. If Colab resets, repeat sections 1–5 before resuming this cell. The run identity prevents a checkpoint from being resumed with another encoder or manifest.


In [ ]:
run_identity = {
    "encoder_sha256": encoder_hash,
    "manifest_sha256": manifest_hash,
    "num_hist": 1,
    "rollout_steps": 1,
    "epochs": 100,
    "batch_size": 8,
}
run_identity_path = RUN_DIR / "run_identity.json"
if run_identity_path.exists():
    assert json.loads(run_identity_path.read_text()) == run_identity, (
        "Drive world-model run belongs to different inputs or settings"
    )
else:
    assert not (RUN_DIR / "last.pt").exists() and not (RUN_DIR / "best.pt").exists(), (
        "Existing world-model checkpoint has no encoder identity; choose a fresh RUN_DIR"
    )
    run_identity_path.write_text(json.dumps(run_identity, indent=2))

train_cmd = [
    sys.executable, "-u", "-m", "src.mylonas_ijepa_wm.wm_proprio",
    "--env", "wall", "--env-config", ENV_CONFIG,
    "--episode-manifest", MANIFEST,
    "--latent-dir", CACHE_DIR / "latents",
    "--trajectory-dir", CACHE_DIR / "trajectories",
    "--output-dir", RUN_DIR,
    "--num-hist", "1", "--rollout-steps", "1",
    "--epochs", "100", "--batch-size", "8",
    "--device", "cuda", "--precision", "auto", "--log-every", "1000",
]
if (RUN_DIR / "last.pt").exists():
    train_cmd.append("--resume")
    print("Resuming world model from", RUN_DIR / "last.pt")
else:
    print("Starting fresh world-model training")
run(train_cmd)


## 7. Inspect the selected world-model checkpoint

In [ ]:
best_wm = RUN_DIR / "best.pt"
last_wm = RUN_DIR / "last.pt"
assert best_wm.is_file() and last_wm.is_file()
for path in (best_wm, last_wm):
    checkpoint = torch.load(path, map_location="cpu", weights_only=True, mmap=True)
    metadata = checkpoint["metadata"]
    assert metadata["experiment_manifest_sha256"] == manifest_hash
    assert metadata["model_configuration"]["num_hist"] == 1
    print(path.name, "epoch:", checkpoint["epoch"] + 1,
          "best validation loss:", checkpoint["best_val_loss"])
    del checkpoint


## 8. Check validation rollouts before planning

This report is local. State prediction error is useful for deciding whether the trained world model is ready for planning; its latent MSE is specific to this encoder.


In [ ]:
rollout_path = RESULTS_DIR / "val_rollouts_h5.json"
run([
    sys.executable, "-u", "-m", "src.mylonas_ijepa_wm.evaluate_rollouts_proprio",
    "--checkpoint", best_wm,
    "--latent-dir", CACHE_DIR / "latents",
    "--trajectory-dir", CACHE_DIR / "trajectories",
    "--split", "val", "--horizon", "5", "--by-wall-layout",
    "--batch-size", "4", "--device", "cuda", "--precision", "auto",
    "--output", rollout_path,
])
report = json.loads(rollout_path.read_text())
for layout, metrics in report["by_wall_layout"].items():
    h1, h5 = metrics["per_horizon"][0], metrics["per_horizon"][4]
    print(layout, "state MSE h1:", round(h1["autoregressive_proprio_mse"], 4),
          "h5:", round(h5["autoregressive_proprio_mse"], 4))


## 9. Smoke check planning on one held-out layout

One episode validates the encoder–world-model pairing and the live DotWall planner. The full run below uses the same action cap, cost, horizon, and CEM settings.


In [ ]:
PLAN_DIR = RESULTS_DIR / "heldout_planning"
PLAN_DIR.mkdir(parents=True, exist_ok=True)
first_layout = manifest["test_layouts"][0]

def layout_config(wall_x, door_y):
    path = PLAN_DIR / f"wall{wall_x}_door{door_y}_env.json"
    layout_settings = {
        **settings,
        "wall_location": wall_x,
        "door_location": door_y,
        "wall_layouts": [[wall_x, door_y]],
    }
    if path.exists():
        assert json.loads(path.read_text()) == layout_settings
    else:
        path.write_text(json.dumps(layout_settings, indent=2))
    return path

def plan_layout(wall_x, door_y, *, num_episodes, output_name, save_videos=False):
    assert json.loads(run_identity_path.read_text())["encoder_sha256"] == encoder_hash, (
        "World model and live planning encoder differ"
    )
    config_path = layout_config(wall_x, door_y)
    output_path = PLAN_DIR / output_name
    command = [
        sys.executable, "-u", "-m", "src.mylonas_ijepa_wm.plan_proprio",
        "--env", "wall", "--env-config", config_path,
        "--encoder-checkpoint", ENCODER_LOCAL,
        "--world-model-checkpoint", best_wm,
        "--output", output_path, "--cost", "position",
        "--num-episodes", str(num_episodes), "--base-seed", "10000",
        "--max-macro-steps", "20", "--planning-horizon", "5",
        "--actions-per-plan", "5", "--population", "300",
        "--num-elites", "30", "--cem-iterations", "10",
        "--candidate-batch-size", "64", "--unbounded-actions",
        "--max-action-length", "3.5", "--device", "cuda",
        "--precision", "auto",
    ]
    if save_videos:
        command += ["--video-dir", PLAN_DIR / f"wall{wall_x}_door{door_y}_videos"]
    run(command)
    summary = json.loads(output_path.read_text())["summary"]
    print(f"Wall ({wall_x}, {door_y}): "
          f"{summary['num_successes']}/{summary['num_episodes']} successes; "
          f"mean final distance {summary['mean_final_goal_distance']:.2f}")
    return output_path

plan_layout(*first_layout, num_episodes=1,
            output_name="smoke_wall%d_door%d.json" % tuple(first_layout),
            save_videos=True)


## 10. Plan on all four unseen test layouts

The four layouts come from the manifest's held-out test split. Each layout uses 20 episodes with the same base seeds, a 3.5 Euclidean primitive-action cap, position cost, horizon 5, and the earlier CEM settings. Outputs remain local; set `SAVE_VIDEOS = True` if you want videos for every run.


In [ ]:
SAVE_VIDEOS = False
planning_rows = []
for wall_x, door_y in manifest["test_layouts"]:
    output = plan_layout(
        wall_x, door_y, num_episodes=20,
        output_name=f"wall{wall_x}_door{door_y}_cap3p5.json",
        save_videos=SAVE_VIDEOS,
    )
    summary = json.loads(output.read_text())["summary"]
    planning_rows.append({
        "wall_layout": [wall_x, door_y],
        "successes": summary["num_successes"],
        "episodes": summary["num_episodes"],
        "mean_final_goal_distance": summary["mean_final_goal_distance"],
    })
print("\nHeld-out planning summary")
for row in planning_rows:
    print(row)
print("Local results:", PLAN_DIR)
